# Module 4: Session Managers

~5 min

Add file-based persistence so the Dussault remembers your research across
restarts. NFL Next Gen Stats stores 10+ years of historical play data in S3 —
every tracking chip coordinate, every route tree, every pressure metric. That
persistence is what makes longitudinal analysis possible. Same principle here:
the agent's conversation history survives process death and picks up where you
left off.

**What you'll build:**
- `FileSessionManager` — persists conversation to disk (survives restarts)
- `SlidingWindowConversationManager` — bounds what the model sees per turn (keeps context manageable)

**Prerequisites:** Modules 1–3 completed, AWS credentials configured (us-west-2)

In [ ]:
!pip install -q strands-agents strands-agents-tools

import sys
sys.path.insert(0, "../shared")
sys.path.insert(0, "../01-agent-loop-tools")

from strands import Agent
from strands.models import BedrockModel
from strands.session.file_session_manager import FileSessionManager
from strands.agent.conversation_manager import SlidingWindowConversationManager
from model_provider import get_model
from dussault_tools import lookup_player, get_game_result, get_season_stats

## Session Persistence

Without a session manager, every `Agent()` instantiation starts from zero — no
memory of prior interactions. That's fine for one-shot queries, but research is
iterative. You ask about Brady's regular season, then drill into his playoff
performance. The agent needs to remember the first answer to give a meaningful
comparison on the second.

`FileSessionManager` writes each message to disk as JSON. On next instantiation
with the same `session_id`, it reloads the full history. The agent picks up
exactly where you left off — even after a process restart.

`SlidingWindowConversationManager` complements this by bounding what the model
actually sees per inference call. History grows on disk; the context window stays
fixed at the most recent N messages.

In [ ]:
import shutil, os

# Start fresh for the demo
if os.path.exists("./sessions"):
    shutil.rmtree("./sessions")

session_manager = FileSessionManager(
    session_id="notebook-demo",
    storage_dir="./sessions",
)

## Create the Persistent Agent

Wire the session manager and conversation manager into the agent. The sliding
window keeps the last 20 messages in context — enough for a multi-turn research
session without blowing the context window on long histories.

In [ ]:
SYSTEM_PROMPT = """You are Dussault, a 2004 New England Patriots Dussault with persistent memory.

If there are previous messages in the conversation history, use that context
to continue the analysis without asking the user to repeat information.

When answering:
- Always look up the data before making claims. Never guess stats.
- Connect facts to story — why something happened matters as much as what happened.
- Reference prior conversation context when relevant.
- Be specific: cite game weeks, scores, stat lines."""

agent = Agent(
    model=get_model(),
    tools=[lookup_player, get_game_result, get_season_stats],
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    session_manager=session_manager,
    callback_handler=None,
)

## First Interaction

Ask about Brady's 2004 season. The agent will use `get_season_stats` to pull
real numbers, then frame the response. This exchange gets persisted to disk.

In [ ]:
result = agent("Tell me about Tom Brady's 2004 season.")
print(result)

## Second Interaction

Now ask a follow-up that depends on the first answer. The agent has the prior
context in its sliding window — it knows we were just discussing Brady's regular
season stats and can compare directly to playoff performance without us repeating
the setup.

In [ ]:
result = agent("How did he do in the playoffs compared to regular season?")
print(result)

## The Persistence Demo

In a notebook this is a single process, so the persistence is invisible — the
agent object is still alive. The real payoff is in the terminal:

```bash
cd samples/04-session-managers
python chat.py --session-id brady-deep-dive
# ask a few questions, then type 'quit'

python chat.py --session-id brady-deep-dive
# the agent remembers everything from the first run
```

Same `session_id` → same conversation restored from `./sessions/` on disk.
Different `session_id` → fresh start. That's the entire persistence model.

In [ ]:
print(f"Session has {len(agent.messages)} messages stored")

## What's Next

The agent is persistent, bounded, and tool-equipped — complete enough to ship.
In **Module 5: Deploy**, you'll package this same agent as a serverless endpoint
via Amazon Bedrock AgentCore Runtime with a single CLI command.